In [1]:
%load_ext autoreload
%autoreload 2

import utils
from feature_engineering import FeatureEngineering


import model
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import feature_engineering
import config  # все пути и настройки проекта лежат в config.py

In [1]:
data_rating = pd.read_csv(config.RATING_CUT_CSV, delimiter=',')
data_movie = pd.read_csv(config.MOVIE_PATH, delimiter=',')

data_rating['ts'] = pd.to_datetime(data_rating['ts'])
data_rating = data_rating.sort_values('ts')

X, y = data_rating.drop(columns=['rating']), data_rating['rating']

print(data_rating)
fe = feature_engineering.FeatureEngineering()
X = fe.transform(X, y)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.7, shuffle=False)
X_valid, X_test, y_valid, y_test = train_test_split(X_valid, y_valid, test_size=0.5, shuffle=False)

        user_id  movie_id  rating                  ts
0         80920     27808     0.5 2015-01-09 02:24:15
1         80920    111617     0.5 2015-01-09 02:24:16
2         80920      7293     0.5 2015-01-09 02:24:17
3         80920      5673     0.5 2015-01-09 02:24:19
4         80920      6287     0.5 2015-01-09 02:24:23
...         ...       ...     ...                 ...
249995    53930    118706     3.5 2015-03-31 06:00:51
249996    16978      2093     3.5 2015-03-31 06:03:17
249997    89081     55232     3.5 2015-03-31 06:11:26
249998    89081     52458     4.0 2015-03-31 06:11:28
249999    87586      7151     3.5 2015-03-31 06:40:02

[250000 rows x 4 columns]


In [3]:
np.random.seed(config.RANDOM_SEED)

users_ids = np.random.choice(a=range(1, 150000), size=1000, replace=False)
data = pd.read_parquet(config.PARQUET_DIR, filters=[('user_id', 'in', users_ids)])

data = data.sort_values(by='ts')

X, y = data.drop(columns=['rating']), data['rating']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, shuffle=False)
X_valid, X_test, y_valid, y_test = train_test_split(X_valid, y_valid, test_size=0.5, shuffle=False)


In [4]:
import recommender
from model import BaseModel
import sqlalchemy
import config

engine = sqlalchemy.create_engine(config.RECSYS_DB_URL)

model = BaseModel(strategy='mean').fit(X_train, y_train)

rec = recommender.Recommender(model=model,
                              user_id=1, n_candidates=config.N_CANDIDATES, engine=engine, random_candidates=True)

In [3]:
from sqlalchemy import create_engine, inspect, text
from sqlalchemy.types import Integer, Float, DateTime, REAL
import config

engine = create_engine(url=config.RECSYS_DB_URL)
l, r = 1, 100
data = pd.read_sql(text('SELECT * '
                           'FROM movie_feats'
                           ' WHERE'
                           ' user_id BETWEEN :le and :ri'), engine, params={'le':l, 'ri':r})

In [6]:
print(data)

       user_id  movie_id  movie_cnt  movie_sum
0           84         1       1532     6512.0
1            8         1       2491    10423.0
2           93         1       3287    13718.0
3           34         1       4250    17665.0
4           13         1       5508    22881.0
...        ...       ...        ...        ...
11096       71    116797          3       11.0
11097       71    117511         10       34.0
11098       71    117590          7       15.0
11099       31    118696        427     1453.0
11100       31    125916         36       58.5

[11101 rows x 4 columns]
